# Chapter 07: Valid Outcomes of Positive Support Size Six

In [13]:
import numpy as np
import itertools
import json
import os
import re
import chipsplitting as cs
from chipsplitting import pairing_matrix, PascalForm, LinearForm
from chipsplitting.hyperfield import HyperfieldVector as HV, HyperfieldHomogeneousLinearSystem as HLinSystem, grid_iter, HyperfieldLinearForm

In [22]:
CONTRACTION_SIZE = 5
DEGREE = 40

## Proposition 7.28

We have $ \lvert \Gamma^{\mathrm{even}}_6 \rvert  = 106806$ and $ \lvert \Gamma^{\mathrm{odd}}_6 \rvert  = 110272$.

In [14]:
def countValidConfigsForContractions(pos_support_size, contraction_size, degree="even", use_extra_constraints = False):
    assert degree == "even" or degree == "odd", "degree must be 'even' or 'odd'"

    d = contraction_size * 3 - 1

    if d % 2 == 0 and degree == "odd":
        d += 1
    elif d % 2 == 1 and degree == "even":
        d += 1
        
    base_types = ["diag", "row", "col"]
    A = [PascalForm(d, b, k) for b in base_types for k in range(contraction_size)] + [PascalForm(d, b, k) for b in base_types for k in range(d - contraction_size + 1, d + 1)]
    
    if use_extra_constraints:
        # (1, d-1) see apple notes 3242444442
        A = A + [PascalForm(d, 'diag', i) - PascalForm(d, 'diag', j) for i,j in [(0,1), (0,2), (0,3), (0,4), (0,d-1), (0,d-2), (0,d-3), (0,d-4), (1,2), (1,3), (1, d), (1,d-1), (1,d-4), (1,d-2), (1,d-3), (2,d), (2,d-1), (2,d-3), (2, d-4), (3,d), (3,d-1), (3,d-2), (3, d-4)]]
        A = A + [PascalForm(d, 'diag', i) - PascalForm(d, 'diag', j) for i,j in [(d-4,d), (d-3,d), (d-2,d), (d-2,d-1), (d-1,d-2), (d-1,d-3), (d-1,d)]]
       
    A = [p.to_hyperfield().contract(contraction_size) for p in A if p != LinearForm.zero(d)]  
    linear_system = HLinSystem(A)
    solutions = linear_system.quick_solve_loop(pos_support_size)
    
    return solutions

In [15]:
%%time
n = 6
contraction_size = 5
res1 = countValidConfigsForContractions(n, contraction_size, "even", use_extra_constraints = True)
print(f"Number of configurations for even d: {len(res1)}")

res2 = countValidConfigsForContractions(n, contraction_size, "odd", use_extra_constraints = True)
print(f"Number of configurations for odd d: {len(res2)}")

Number of configurations for even d: 106806
Number of configurations for odd d: 110272
CPU times: user 1.42 s, sys: 9.7 ms, total: 1.43 s
Wall time: 1.43 s


## Step 3: Filtering Invalid Cases


### Load fixed-contractable Pascal forms

In [17]:
# contains all fixed-contractable forms
DATA = []

def list_files_in_directory(directory):
    files = [f for f in os.listdir(directory) if os.path.isfile(os.path.join(directory, f))]
    return files

def strip_npy_extension(file):
    return file[:-len(".npy")]

directory_path = 'filter' 
files = sorted(list_files_in_directory(directory_path), key=len)

for file in files:
    name = strip_npy_extension(file)
    DATA.append((name, np.load(f"filter/{file}")))
        
print("Data successfully loaded")

Data successfully loaded


## Code 

In [18]:
def absolute(deg, c):
    if type(c) is int:
        return c

    if type(c) is np.str_:
        c = str(c)

    if type(c) is str:
        if len(c) == 3:
            subtrahend = int(c[2])
            if subtrahend >= CONTRACTION_SIZE:
                raise Exception(f"Invalid subtrahend: {c}")
            return deg - int(c[2])
        elif c == 'd':
            return deg
        elif len(c) == 1:
            return int(c)
        else:
            raise Exception(f'string {c} has not length 3')

    raise Exception(f"Invalid type. {c} is of type {type(c)}")

def rel(degree, index):
    assert index < CONTRACTION_SIZE or index > degree - CONTRACTION_SIZE
    return f"d-{degree - index}" if index > CONTRACTION_SIZE else index

def parse_expression(expression):
    if expression[0] != '-':
        expression = '+' + expression
        
    ops = re.findall(r'[\+|-]',expression)
    summands = re.split(r'[\+|-]', expression[1:])
    return (summands, ops)

def realize_expression(degree, mode, op, unit):
    if op == '+':
        return PascalForm(degree, mode, unit)
    elif op == '-':
        return -PascalForm(degree, mode, unit)
    else:
        raise Exception(degree, mode, op, unit)

def form_from_expression(degree, expression, abs_units):
    summands, ops = parse_expression(expression)
    form = LinearForm.zero(degree)
    for mode, op, unit in zip(summands, ops, abs_units):
        form = form + realize_expression(degree, mode, op, unit)
    return form


def hyperfield_vector_from_support(d, support_pos, support_neg):
    w = [-1] + [0] * (d-1)
    for x in support_pos:
        w[x] = 1
    return HV(w)

def is_root(hyperfield_forms, w):
    for p in hyperfield_forms:
        y = p(w)
        if not np.isnan(y) and y != 0:
            return False
    return True

### Build Filter

In [21]:
%%time
d = DEGREE
FILTER = set()

for expression, units_list in DATA:
    modes, ops = parse_expression(expression)
    for rel_units in units_list:
        units = tuple([absolute(d, unit) for unit in rel_units])
        p = form_from_expression(d, expression, units)

        if p != LinearForm.zero(p.degree) and not p.support_pos[0] and not p.support_neg[0]:
            FILTER.add(p.to_hyperfield().contract(contraction_size))

len(FILTER)

CPU times: user 35min 27s, sys: 2.3 s, total: 35min 30s
Wall time: 35min 30s


18273

In [13]:
%%time
import time

# HERE CURRENT

VALID_SUPPORTS = []
TO_FILTER = res1
CUSTOM_FILTER = [p for p in FILTER if not p.support_pos[0] and not p.support_neg[0]]

print(f"{len(TO_FILTER)} supports filter")
print(f"CUSTOM FILTER length is {len(CUSTOM_FILTER)}")

start = time.time()
for i, support_pos in enumerate(TO_FILTER):
    if i == 1000:
        end = time.time()
        elapsed = end - start
        print(f"Elapsed: {elapsed}. Estimate: {len(TO_FILTER) / 1000 * elapsed} seconds.")
    w = hyperfield_vector_from_support(contraction_size ** 2 * 3 + CONTRACTION_SIZE * 4, support_pos, [])
    if is_root(CUSTOM_FILTER, w):
        VALID_SUPPORTS.append(support_pos)

print(f"reduced {len(TO_FILTER)} supports to {len(VALID_SUPPORTS)} supports")

106806 supports filter
CUSTOM FILTER length is 18273
Elapsed: 50.45843195915222. Estimate: 5389.263283829212 seconds.
reduced 106806 supports to 6700 supports
CPU times: user 1h 34min 15s, sys: 12.7 s, total: 1h 34min 28s
Wall time: 1h 34min 29s


In [19]:
%%time
d = DEGREE + 1
FILTER = set()

for expression, units_list in DATA:
    modes, ops = parse_expression(expression)
    for rel_units in units_list:
        units = tuple([absolute(d, unit) for unit in rel_units])
        p = form_from_expression(d, expression, units)

        if p != LinearForm.zero(p.degree) and not p.support_pos[0] and not p.support_neg[0]:
            FILTER.add(p.to_hyperfield().contract(contraction_size))

len(FILTER)

CPU times: user 37min 17s, sys: 2.18 s, total: 37min 19s
Wall time: 37min 19s


18034

In [20]:
%%time
import time

# HERE CURRENT

VALID_SUPPORTS = []
TO_FILTER = res2
CUSTOM_FILTER = [p for p in FILTER if not p.support_pos[0] and not p.support_neg[0]]

print(f"{len(TO_FILTER)} supports filter")
print(f"CUSTOM FILTER length is {len(CUSTOM_FILTER)}")

start = time.time()
for i, support_pos in enumerate(TO_FILTER):
    if i == 1000:
        end = time.time()
        elapsed = end - start
        print(f"Elapsed: {elapsed}. Estimate: {len(TO_FILTER) / 1000 * elapsed} seconds.")
    w = hyperfield_vector_from_support(contraction_size ** 2 * 3 + CONTRACTION_SIZE * 4, support_pos, [])
    if is_root(CUSTOM_FILTER, w):
        VALID_SUPPORTS.append(support_pos)

print(f"reduced {len(TO_FILTER)} supports to {len(VALID_SUPPORTS)} supports")

110272 supports filter
CUSTOM FILTER length is 18034
Elapsed: 52.29708409309387. Estimate: 5766.9040571136475 seconds.
reduced 110272 supports to 8737 supports
CPU times: user 1h 36min 28s, sys: 3.83 s, total: 1h 36min 32s
Wall time: 1h 36min 32s
